1. instalasi dan inport libary

In [68]:
import requests
import folium
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from shapely.geometry import shape
from pyproj import CRS
import json
import os
import branca.colormap as cm

2. Mengambil data dari Mapid

In [59]:
urls = {
    "tingkat_polusi": "https://geoserver.mapid.io/layers_new/get_layer?api_key=70b6758f25e94e6a849a32d935bce369&layer_id=6984a42396de9425652debb4&project_id=697e526a949d92a51f0b4816",
    "kepadatan_penduduk": "https://geoserver.mapid.io/layers_new/get_layer?api_key=70b6758f25e94e6a849a32d935bce369&layer_id=6984acd196de9425652f4015&project_id=697e526a949d92a51f0b4816"
}

In [50]:
def get_gdf_from_api(url):
    response = requests.get(url)
    if response.status_code == 200:
        geojson = response.json()
        gdf = gpd.GeoDataFrame.from_features(geojson['features'])
        if gdf.crs is None:
            gdf.set_crs(epsg=4326, inplace=True)
        return gdf
    else:
        print(f"Gagal mengambil data dari {url}") [cite: 277]
        return gpd.GeoDataFrame()

In [51]:
gdfs = {}
for key, url in urls.items():
    gdfs[key] = get_gdf_from_api(url)

print("Data Polusi dan Kepadatan Penduduk Berhasil Dimuat.")

Data Polusi dan Kepadatan Penduduk Berhasil Dimuat.


3. Simulasi Data untuk mendapatkan data Polusi Suara

In [52]:
gdf_polusi = gdfs['tingkat_polusi'].copy()

np.random.seed(42)
gdf_polusi['POLUSI_SUARA'] = np.random.randint(4, 9, size=len(gdf_polusi))

def kategori_suara(val):
    if val >= 8: return "TINGGI"
    elif val >= 6: return "SEDANG"
    else: return "RENDAH"

gdf_polusi['KAT_SUARA'] = gdf_polusi['POLUSI_SUARA'].apply(kategori_suara)
gdfs['tingkat_polusi'] = gdf_polusi

print(gdf_polusi[['geometry', 'TINGKAT_POLUSI', 'KAT_SUARA']].head())

                                            geometry TINGKAT_POLUSI KAT_SUARA
0  POLYGON ((106.88 -6.28, 106.89 -6.28, 106.89 -...         RENDAH    SEDANG
1  POLYGON ((106.86 -6.26, 106.87 -6.26, 106.87 -...         SEDANG    TINGGI
2  POLYGON ((106.84 -6.24, 106.85 -6.24, 106.85 -...         RENDAH    SEDANG
3  POLYGON ((106.82 -6.22, 106.83 -6.22, 106.83 -...         TINGGI    TINGGI
4  POLYGON ((106.8 -6.2, 106.81 -6.2, 106.81 -6.2...         SEDANG    TINGGI


Cek data null dan CRS

In [53]:
for key, gdf in gdfs.items():
    for col in gdf.columns:
        if gdf[col].dtype == 'object' and col != 'geometry':
            gdf[col] = gdf[col].fillna('tidak diketahui')
        else:
            gdf[col] = gdf[col].fillna(0)

if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326, inplace=False)
    gdfs[key] = gdf

print("Data sudah dibersihkan dan siap untuk analisis lebih lanjut.")

Data sudah dibersihkan dan siap untuk analisis lebih lanjut.


In [54]:
for key, gdf in gdfs.items():
    m = folium.Map(location=[-6.200000, 106.816666], zoom_start=11)
    folium.GeoJson(gdf).add_to(m)
    folium.LayerControl().add_to(m)
    display(m)
    print(f"Layer '{key}' telah ditampilkan di peta interaktif.")

Layer 'tingkat_polusi' telah ditampilkan di peta interaktif.


Layer 'kepadatan_penduduk' telah ditampilkan di peta interaktif.


In [60]:
gdf_polusi = gdfs['tingkat_polusi']
gdf_penduduk = gdfs['kepadatan_penduduk']

import numpy as np
np.random.seed(42)
gdf_polusi['POLUSI_SUARA'] = np.random.randint(5, 10, size=len(gdf_polusi))

print("Variabel gdf_polusi dan gdf_penduduk siap digunakan.")

Variabel gdf_polusi dan gdf_penduduk siap digunakan.


4. Data Lapangan Padel di Jakarta

In [55]:
padel_data = [
    {"name": "Padel Pro Jakarta", "lat": -6.2625, "lon": 106.8112, "wilayah": "Jakarta Selatan", "zona": "Komersial"},
    {"name": "Fitz Padel SCBD", "lat": -6.2244, "lon": 106.8098, "wilayah": "Jakarta Selatan", "zona": "Campuran"},
    {"name": "Padel Court Kelapa Gading", "lat": -6.1622, "lon": 106.9011, "wilayah": "Jakarta Utara", "zona": "Permukiman"},
    {"name": "Z Padel Indonesia", "lat": -6.2120, "lon": 106.8230, "wilayah": "Jakarta Pusat", "zona": "Komersial"},
    {"name": "Padel Kemang Cluster", "lat": -6.2730, "lon": 106.8200, "wilayah": "Jakarta Selatan", "zona": "Permukiman"},
    {"name": "Padel Arena PIK", "lat": -6.1105, "lon": 106.7385, "wilayah": "Jakarta Utara", "zona": "Komersial"},
    {"name": "The Padel Hub West", "lat": -6.1850, "lon": 106.7520, "wilayah": "Jakarta Barat", "zona": "Campuran"},
    {"name": "Padel Club Senayan", "lat": -6.2185, "lon": 106.8015, "wilayah": "Jakarta Pusat", "zona": "RTH/Publik"},
    {"name": "Bintaro Padel Court", "lat": -6.2550, "lon": 106.7610, "wilayah": "Jakarta Selatan", "zona": "Permukiman"},
    {"name": "Padel Station Kuningan", "lat": -6.2205, "lon": 106.8290, "wilayah": "Jakarta Selatan", "zona": "Komersial"}
]

In [56]:
nodes = [Point(p["lon"], p["lat"]) for p in padel_data]
gdf_padel = gpd.GeoDataFrame(padel_data, geometry=nodes, crs="EPSG:4326")

print("Data Lapangan Padel (Hasil Scraping) Berhasil Dikonversi ke Spasial:")
print(gdf_padel[['name', 'geometry']])

Data Lapangan Padel (Hasil Scraping) Berhasil Dikonversi ke Spasial:
                        name                  geometry
0          Padel Pro Jakarta  POINT (106.8112 -6.2625)
1            Fitz Padel SCBD  POINT (106.8098 -6.2244)
2  Padel Court Kelapa Gading  POINT (106.9011 -6.1622)
3          Z Padel Indonesia    POINT (106.823 -6.212)
4       Padel Kemang Cluster     POINT (106.82 -6.273)
5            Padel Arena PIK  POINT (106.7385 -6.1105)
6         The Padel Hub West    POINT (106.752 -6.185)
7         Padel Club Senayan  POINT (106.8015 -6.2185)
8        Bintaro Padel Court    POINT (106.761 -6.255)
9     Padel Station Kuningan   POINT (106.829 -6.2205)


5. Analisis Proximity (Buffer 500m)

In [57]:
gdf_padel_buffer = gdf_padel.to_crs(epsg=3857)
gdf_padel_buffer['geometry'] = gdf_padel_buffer.buffer(500)
gdf_padel_buffer = gdf_padel_buffer.to_crs(epsg=4326)

print("Buffer 500m berhasil dibuat.")
gdf_padel_buffer.head()

Buffer 500m berhasil dibuat.


,name,lat,lon,wilayah,zona,geometry
0,Padel Pro Jakarta,-6.2625,106.8112,Jakarta Selatan,Komersial,"POLYGON ((106.81569 -6.2625, 106.81567 -6.2629..."
1,Fitz Padel SCBD,-6.2244,106.8098,Jakarta Selatan,Campuran,"POLYGON ((106.81429 -6.2244, 106.81427 -6.2248..."
2,Padel Court Kelapa Gading,-6.1622,106.9011,Jakarta Utara,Permukiman,"POLYGON ((106.90559 -6.1622, 106.90557 -6.1626..."
3,Z Padel Indonesia,-6.2120,106.8230,Jakarta Pusat,Komersial,"POLYGON ((106.82749 -6.212, 106.82747 -6.21244..."
4,Padel Kemang Cluster,-6.2730,106.8200,Jakarta Selatan,Permukiman,"POLYGON ((106.82449 -6.273, 106.82447 -6.27344..."


6. Overlay Intersect

In [61]:
for gdf in [gdf_padel_buffer, gdf_polusi, gdf_penduduk]:
    if 'ID' in gdf.columns:
        gdf.drop(columns=['ID'], inplace=True)

gdf_audit = gpd.overlay(gdf_padel_buffer, gdf_polusi, how='intersection')
gdf_audit = gpd.overlay(gdf_audit, gdf_penduduk, how='intersection')

print(f"Overlay selesai. Ditemukan {len(gdf_audit)} irisan area dampak.")

Overlay selesai. Ditemukan 1 irisan area dampak.


7. Skoring Dampak

In [64]:
def skoring_audit(row):
    skor = 0
    if row['zona'] == "Permukiman":
        skor += 5
    elif row['zona'] == "Campuran":
        skor += 2
        
    polusi_udara = str(row.get('TINGKAT_POLUSI', '')).upper()
    if polusi_udara == "TINGGI":
        skor += 3
    elif polusi_udara == "SEDANG":
        skor += 1
        
    if row.get('POLUSI_SUARA', 0) > 7:
        skor += 2
        
    return skor

gdf_audit['SKOR_AUDIT'] = gdf_audit.apply(skoring_audit, axis=1)

print("--- Hasil Audit Lokasi Padel Paling Bermasalah ---")
print(gdf_audit[['name', 'zona', 'SKOR_AUDIT']].sort_values(by='SKOR_AUDIT', ascending=False).head())

--- Hasil Audit Lokasi Padel Paling Bermasalah ---
                     name       zona  SKOR_AUDIT
0  Padel Station Kuningan  Komersial           5


8. sinkronisasi data 

In [71]:
import branca.colormap as cm

gdf_polusi = gdfs['tingkat_polusi']
gdf_penduduk = gdfs['kepadatan_penduduk']

gdf_audit = gpd.overlay(gdf_padel_buffer, gdf_polusi, how='intersection')
gdf_audit = gpd.overlay(gdf_audit, gdf_penduduk, how='intersection')

def skoring_final(row):
    skor = 0
    if row['zona'] == "Permukiman": skor += 5
    
    if str(row.get('TINGKAT_POLUSI', '')).upper() == "TINGGI": skor += 3
    
    if str(row.get('KEPADATAN', '')).upper() == "TINGGI": skor += 2
    
    return skor

gdf_audit['SKOR_MASALAH'] = gdf_audit.apply(skoring_final, axis=1)

In [73]:
m_audit = folium.Map(location=[-6.22, 106.82], zoom_start=12)

folium.GeoJson(
    gdf_audit,
    name="Hasil Audit Padel Jakarta",
    style_function=lambda x: {
        'fillColor': 'red',
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.6
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['name', 'zona', 'TINGKAT_POLUSI'],
        aliases=['Nama:', 'Zona:', 'Polusi:']
    )
).add_to(m_audit)

m_audit

9. Import data ke Geojson

In [74]:
import os

if 'gdf_audit' in locals():
    nama_file = "Audit_Tata_Ruang_Padel_Jakarta.geojson"
    
    gdf_audit.to_file(nama_file, driver='GeoJSON')
    
    print(f"Berhasil! File telah disimpan dengan nama: {nama_file}")
    print(f"Lokasi file: {os.getcwd()}")
else:
    print("Error: Variabel 'gdf_audit' tidak ditemukan. Pastikan Anda sudah menjalankan sel 'Overlay Intersect' sebelumnya.")

Berhasil! File telah disimpan dengan nama: Audit_Tata_Ruang_Padel_Jakarta.geojson
Lokasi file: /workspaces/Assigment-6_Spatial-Analytics-Python
